# 01 · 张量（Tensor）基础

> 张量是 PyTorch 的**唯一**数据结构。掌握张量 = 掌握 PyTorch 的一半。
> 类比：张量 ≈ NumPy 的 `ndarray`，但**支持 GPU 加速和自动求导**。

## 1.1 维度概念

| 维度 | 名称 | 例子 |
|---|---|---|
| 0 | 标量 scalar | `3.14` |
| 1 | 向量 vector | `[1, 2, 3]` |
| 2 | 矩阵 matrix | `[[1,2],[3,4]]` |
| 3+ | 张量 tensor | 彩色图片 `[C, H, W]`，一批图片 `[N, C, H, W]` |

深度学习中常见的形状写法：
- `(N,)`：N 个标量
- `(N, C)`：N 个样本、每个 C 个特征
- `(N, C, H, W)`：一批图片（批大小、通道、高、宽）
- `(N, T, D)`：一批序列（批大小、时间步、特征维度）

导入约定（本套教程统一）：

In [ ]:
import torch
import numpy as np

torch.manual_seed(0)          # 固定随机种子，保证结果可复现
print(torch.__version__)

## 1.2 创建张量

### 从 Python 列表创建

In [ ]:
# 从列表创建（会自动推断 dtype）
a = torch.tensor([1, 2, 3])
b = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("1维:", a, "| shape:", a.shape, "| dtype:", a.dtype)
print("2维:\n", b, "| shape:", b.shape, "| dtype:", b.dtype)

# 指定 dtype
c = torch.tensor([1, 2, 3], dtype=torch.float32)
print(c, c.dtype)

# 从 numpy 创建（共享内存！改一个另一个也变）
n = np.array([[1, 2], [3, 4]], dtype=np.float32)
t = torch.from_numpy(n)
print("from_numpy:", t)
n[0, 0] = 99
print("改 numpy 后 tensor 也变了:", t)

### 特殊值张量

In [ ]:
print("全 0 :", torch.zeros(2, 3))
print("全 1 :", torch.ones(2, 3))
print("单位阵 :", torch.eye(3))
print("填充 :", torch.full((2, 3), 7.0))
print("未初始化（内容随机）:", torch.empty(2, 2))
print("跟随其它张量的形状:", torch.zeros_like(torch.ones(2, 3)))

### 序列与网格

In [ ]:
print("arange(0,10,2) :", torch.arange(0, 10, 2))          # 不含终点，步长 2
print("arange(5)      :", torch.arange(5))
print("linspace(0,1,5):", torch.linspace(0, 1, 5))         # 含终点，共 5 个数
print("logspace       :", torch.logspace(0, 3, 4))         # 10^0 ... 10^3

# 网格（常用于画图）
x = torch.tensor([1, 2, 3])
y = torch.tensor([10, 20])
gx, gy = torch.meshgrid(x, y, indexing="ij")
print("meshgrid x:\n", gx, "\nmeshgrid y:\n", gy)

### 随机张量

| 函数 | 分布 |
|---|---|
| `torch.rand` | [0,1) 均匀分布 |
| `torch.randn` | 标准正态 `N(0,1)` |
| `torch.randint` | 整数均匀分布 |
| `torch.randperm` | 0..n-1 的随机排列（打乱用） |
| `torch.normal(mu, sigma)` | 自定义正态 |

In [ ]:
print("rand(2,3):\n", torch.rand(2, 3))
print("randn(2,3):\n", torch.randn(2, 3))
print("randint(0,10,(2,3)):\n", torch.randint(0, 10, (2, 3)))
print("randperm(10):", torch.randperm(10))
print("normal(mean=5,std=2):", torch.normal(mean=5.0, std=2.0, size=(5,)))

## 1.3 张量属性

```python
x.shape      # 形状，等价于 x.size()
x.dtype      # 数据类型
x.device     # 所在设备
x.ndim       # 维度数 (等价 len(x.shape))
x.numel()    # 元素总数
x.T          # 转置 (2维及以下)
x.is_cuda    # 是否在 GPU
```

In [ ]:
x = torch.randn(2, 3, 4)
print("shape   :", x.shape)
print("size()  :", x.size())
print("ndim    :", x.ndim)
print("dtype   :", x.dtype)
print("device  :", x.device)
print("numel   :", x.numel())
print("前两个维度:", x.shape[:2])

## 1.4 数据类型与转换

深度学习最常用的两种：
- `torch.float32`（简称 float）：网络权重、输入特征
- `torch.int64`（简称 long）：分类标签（类别索引）

**标签必须是 long，输入必须是 float**，否则 `CrossEntropyLoss` 会报错。

In [ ]:
x = torch.tensor([1.7, 2.3])
print("原 dtype:", x.dtype)

print("转 int64 :", x.long(), x.long().dtype)
print("转 int32 :", x.int(), x.int().dtype)
print("转 float64:", x.double().dtype)
print("四舍五入取整:", torch.round(x))
print("向下取整:", torch.floor(x), " 向上:", torch.ceil(x))

# 布尔张量
m = torch.tensor([True, False, True])
print("bool:", m.dtype)
print("bool 转 float:", m.float())

### 设备转换

```python
x.to("cuda")       # 到 GPU
x.cpu()            # 回 CPU
x.to(device)       # 用统一的 device 变量（推荐）
```
**注意**：不同设备上的张量不能直接运算。

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(2, 3)
print("CPU 张量:", x.device)

x2 = x.to(device)
print("移动后   :", x2.device)

# 张量之间运算要求 dtype/device 一致
if torch.cuda.is_available():
    y = torch.randn(2, 3, device="cuda")
    print("GPU 相加 ok:", (x2 + y).device)
else:
    print("当前无 GPU，跳过设备运算演示")

## 1.5 索引与切片

语法和 NumPy **完全一致**。

In [ ]:
x = torch.arange(12).reshape(3, 4)
print("x =\n", x)

print("第 0 行      :", x[0])
print("第 1 行第 2 列:", x[1, 2])
print("前两行       :\n", x[:2])
print("最后一列     :", x[:, -1])
print("第 0 行第 1 列后:", x[0, 1:])
print("步长切片     :\n", x[::2, ::2])

# 布尔掩码
print("大于 5 的元素:", x[x > 5])

# 花式索引
idx = torch.tensor([0, 2])
print("取第 0、2 行:\n", x[idx])

# 用索引赋值
y = x.clone()          # 复制一份，避免改到 x
y[0, 0] = 999
print("赋值后 y[0]:", y[0], "| 原 x[0] 不变:", x[0])

> **易错点**：切片是**视图(view)**，不是复制！修改切片会影响原张量。
> 需要复制用 `.clone()`。

```python
z = x[0]        # 视图
z[0] = 100      # x 也会被改！
```

In [ ]:
x = torch.arange(6).reshape(2, 3)
z = x[0]           # 视图！
z[0] = 100
print("x 被修改了:\n", x)

x = torch.arange(6).reshape(2, 3)
z = x[0].clone()   # 复制
z[0] = 100
print("x 不受影响:\n", x)

## 1.6 形状变换（重点）

| 方法 | 作用 |
|---|---|
| `view` / `reshape` | 改变形状（元素数不变） |
| `flatten` | 展平 |
| `squeeze` | 去掉长度为 1 的维度 |
| `unsqueeze` | 增加一个长度为 1 的维度 |
| `permute` | 任意交换维度顺序 |
| `transpose` | 交换两个维度 |
| `expand` / `repeat` | 复制扩展 |

`view` 要求内存连续，`reshape` 会自动处理（推荐新手用 `reshape`）。

In [ ]:
x = torch.arange(12)
print("原始:", x)

print("reshape(3,4):\n", x.reshape(3, 4))
print("reshape(2,-1):\n", x.reshape(2, -1))     # -1 自动推断
print("view(4,3):\n", x.view(4, 3))

In [ ]:
# squeeze / unsqueeze
x = torch.randn(1, 3, 1, 5)
print("原始形状:", x.shape)
print("squeeze 后:", x.squeeze().shape)          # 去掉所有长度 1 的维度
print("squeeze(0) 后:", x.squeeze(0).shape)
print("squeeze(2) 后:", x.squeeze(2).shape)

y = torch.randn(3, 5)
print("\n3x5 -> unsqueeze(0):", y.unsqueeze(0).shape)   # (1,3,5)
print("3x5 -> unsqueeze(1):", y.unsqueeze(1).shape)     # (3,1,5)
print("3x5 -> unsqueeze(-1):", y.unsqueeze(-1).shape)   # (3,5,1)

In [ ]:
# permute / transpose（图像常用）
img = torch.randn(3, 32, 32)          # (C, H, W)
print("原图形状 (C,H,W):", img.shape)

hwc = img.permute(1, 2, 0)            # -> (H, W, C)，画图时需要
print("permute 后 (H,W,C):", hwc.shape)

print("transpose(0,1):", img.transpose(0, 1).shape)

# flatten：把 H,W 拉平
print("flatten:", img.reshape(3, -1).shape)

In [ ]:
# expand / repeat：扩展
x = torch.tensor([[1], [2], [3]])     # (3,1)
print("expand 到 (3,4):\n", x.expand(3, 4))       # 广播复制，不占新内存
print("repeat 到 (3,4):\n", x.repeat(1, 4))       # 真实复制数据

## 1.7 拼接与拆分

| 方法 | 作用 |
|---|---|
| `torch.cat([a,b], dim)` | 在已有维度上拼接（不新增维度） |
| `torch.stack([a,b], dim)` | 在新维度上堆叠（新增维度） |
| `torch.split(x, size, dim)` | 按大小拆分 |
| `torch.chunk(x, n, dim)` | 平均拆成 n 份 |

In [ ]:
a = torch.tensor([[1, 2]])
b = torch.tensor([[3, 4]])

print("cat dim=0:\n", torch.cat([a, b], dim=0))     # (2,2)
print("cat dim=1:\n", torch.cat([a, b], dim=1))     # (1,4)
print("stack dim=0 形状:", torch.stack([a, b], dim=0).shape)  # (2,1,2)
print("stack dim=1 形状:", torch.stack([a, b], dim=1).shape)  # (1,2,2)

In [ ]:
x = torch.arange(10)
print("split 成 [3,3,4]:", [p.tolist() for p in torch.split(x, [3, 3, 4])])
print("split 每块 4:", [p.tolist() for p in torch.split(x, 4)])

x = torch.arange(12).reshape(3, 4)
print("chunk 成 3 份:")
for i, c in enumerate(torch.chunk(x, 3, dim=0)):
    print(" 块", i, c.squeeze().tolist())

## 1.8 广播机制（Broadcasting）

当两个张量形状不同但满足规则时，PyTorch 会自动"扩展"较小的一方。

**规则**（从最后一维往前比较）：
1. 维度相等 → 可以
2. 其中一个是 1 → 可以（扩展成另一个）
3. 其中一个不存在 → 补 1 后按上面处理
4. 否则 → 报错

例：`(3,1)` 与 `(1,4)` → `(3,4)`

In [ ]:
a = torch.tensor([[1], [2], [3]])      # (3,1)
b = torch.tensor([[10, 20, 30, 40]])   # (1,4)
print("a+b 形状:", (a + b).shape)
print(a + b)

# 常见：给每行加不同偏置
x = torch.randn(4, 5)                    # 4 个样本，5 个特征
bias = torch.tensor([1., 2., 3., 4., 5.])  # 每个特征一个偏置
print("\n广播加偏置:", (x + bias).shape)

## 1.9 数学运算

### 逐元素运算

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("a + b :", a + b)
print("a - b :", a - b)
print("a * b :", a * b)      # 逐元素
print("a / b :", a / b)
print("a ** 2:", a ** 2)
print("a % b :", a % b)

# 数学函数
x = torch.tensor([0.0, 0.5, 1.0])
print("\nexp   :", torch.exp(x))
print("log   :", torch.log(x + 1))     # 避免 log(0)
print("sqrt  :", torch.sqrt(x))
print("sin   :", torch.sin(x))
print("abs   :", torch.abs(torch.tensor([-1., 2.])))
print("clamp :", torch.clamp(torch.tensor([-2., 0.5, 5.]), 0, 1))

### 矩阵乘法

用 `@` 或 `torch.matmul`。形状规则：`(n, m) @ (m, p) = (n, p)`

In [ ]:
A = torch.randn(3, 4)
B = torch.randn(4, 2)
print("A@B:", (A @ B).shape)                 # (3,2)
print("matmul:", torch.matmul(A, B).shape)
print("等价 mm:", torch.mm(A, B).shape)      # mm 只支持 2 维

# 批量矩阵乘法 bmm / matmul
X = torch.randn(8, 3, 4)     # 8 个 3x4 矩阵
Y = torch.randn(8, 4, 5)
print("批量:", torch.matmul(X, Y).shape)     # (8,3,5)

# 广播矩阵乘（注意力机制里常见）：Z 形状 (4,5)，广播到每个批量
Z = torch.randn(4, 5)
print("broadcast matmul:", torch.matmul(X, Z).shape)  # (8,3,5)

### 归约运算（极其常用）

关键参数 `dim`：沿哪个维度归约（该维度消失，除非 `keepdim=True`）。

In [ ]:
x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])       # (2,3)

print("sum 全部   :", x.sum())
print("sum dim=0  :", x.sum(dim=0))     # 按列求和 -> (3,)
print("sum dim=1  :", x.sum(dim=1))     # 按行求和 -> (2,)
print("keepdim    :", x.sum(dim=1, keepdim=True).shape)

print("\nmean 全部:", x.mean(), " mean dim=0:", x.mean(dim=0))
print("max 全部 :", x.max())
print("max dim=1:", x.max(dim=1))       # 返回 (values, indices) 两个张量
print("argmax dim=1:", x.argmax(dim=1)) # 返回下标

print("\nstd :", x.std())
print("prod:", x.prod())
print("min :", x.min(dim=1).values)

> **重要**：分类任务中 `argmax(dim=1)` 用来把"每个类别的分数"变成"预测类别"。

### 累积与差分

In [ ]:
x = torch.arange(1, 6)
print("cumsum :", x.cumsum(0))
print("cumprod:", x.cumprod(0))
print("diff   :", torch.diff(x))

## 1.10 比较与条件

| 函数 | 作用 |
|---|---|
| `torch.eq / ne / gt / lt` | 等于/不等于/大于/小于 |
| `torch.allclose(a,b)` | 近似相等（浮点比较） |
| `torch.where(cond, a, b)` | 条件选择 |
| `torch.clamp` | 截断 |
| `torch.sort` | 排序 |
| `torch.topk` | 取最大的 k 个 |

In [ ]:
a = torch.tensor([1., 2., 3.])
b = torch.tensor([3., 2., 1.])

print("a > b :", a > b)
print("a == b:", a == b)
print("where :", torch.where(a > b, a, b))   # 逐元素取较大的

x = torch.tensor([0.1, 0.2, 0.9])
print("allclose:", torch.allclose(x, torch.tensor([0.1, 0.2, 0.9])))

# topk：取最大的 2 个（常用于 Top-k 准确率）
scores = torch.tensor([0.1, 0.7, 0.5, 0.9])
v, i = torch.topk(scores, k=2)
print("top2 values:", v, " indices:", i)

# sort
v, i = torch.sort(scores, descending=True)
print("sorted:", v, i)

## 1.11 与 NumPy 互转

| 方向 | 方法 |
|---|---|
| NumPy → Tensor | `torch.from_numpy(np)`（共享内存） |
| NumPy → Tensor | `torch.tensor(np)`（复制数据） |
| Tensor → NumPy | `t.numpy()`（CPU 上共享内存） |
| Tensor → NumPy | `t.cpu().numpy()`（GPU 张量必须先回 CPU） |

In [ ]:
a = np.array([1.0, 2.0, 3.0])

t1 = torch.from_numpy(a)     # 共享内存
t2 = torch.tensor(a)         # 复制

a[0] = 100
print("from_numpy 跟随变化:", t1)
print("tensor 不变       :", t2)

back = t2.numpy()
print("转回 numpy:", back, type(back))

## 1.12 综合练习

**练习 1**：创建一个形状为 `(4, 3, 32, 32)` 的随机张量（模拟一批 4 张 3 通道 32x32 图片），
把它变成 `(4, 32, 32, 3)`（用于画图）。

**练习 2**：生成 100 个样本、每个样本 3 个特征的随机数据，
用广播给每个特征分别加上偏置 `[1, -1, 0.5]`，然后按特征维度求均值。

**练习 3**：给定 `logits = torch.randn(5, 10)`（5 个样本、10 个类别），
求出每个样本预测的类别、以及每个样本的最大值。

先自己写，再看参考答案。

In [ ]:
# 练习 1 参考答案
x = torch.randn(4, 3, 32, 32)
y = x.permute(0, 2, 3, 1)
print("练习1:", y.shape)

# 练习 2
data = torch.randn(100, 3)
bias = torch.tensor([1.0, -1.0, 0.5])
out = data + bias                # 广播
print("练习2 均值:", out.mean(dim=0))

# 练习 3
logits = torch.randn(5, 10)
pred = logits.argmax(dim=1)
mx = logits.max(dim=1).values
print("练习3 预测类别:", pred)
print("练习3 最大值  :", mx)

## 1.13 小结

- 张量是 PyTorch 的基础数据结构，用法接近 NumPy。
- 常用形状：`(N, C, H, W)` 图片、`(N, T, D)` 序列、`(N, C)` 表格。
- 关键操作：创建、`reshape`、索引、广播、`matmul`、归约、拼接。
- **标签是 long，输入和权重是 float**，跨设备要 `.to(device)`。
- 切片是视图，要复制用 `.clone()`。

下一节：`02_自动求导Autograd.ipynb`